# 01 — Data Collection

Kumpulkan semua sumber data mentah. Semua kode ada inline di notebook ini.

| Sumber | Script asal | Output |
|--------|------------|--------|
| OSM Overpass | `collect_osm.py` | `data/raw/venues_osm_raw.csv` |
| Massive-STEPS (HuggingFace) | `collect_steps.py` | `data/raw/jakarta_checkins_raw.csv` |
| Google Places venue wisata | `collect_venues_google.py` | `data/raw/venues_google_raw.csv` |
| Google Places hotel | `collect_hotels_google.py` | `data/processed/hotels_google.csv` |

> **Catatan**: Sel `[RUN]` butuh koneksi internet + `GOOGLE_PLACES_KEY` di env.  
> Kalau data sudah ada, langsung lompat ke sel `[STATS]`.

In [ ]:
import sys, os
sys.stdout.reconfigure(encoding='utf-8', errors='replace')

ROOT = os.path.abspath(os.path.join(os.getcwd(), '../..'))
sys.path.insert(0, ROOT)
os.chdir(ROOT)

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 110
matplotlib.rcParams['axes.spines.top'] = False
matplotlib.rcParams['axes.spines.right'] = False

import config
print(f'Root: {ROOT}')
print(f'GOOGLE_PLACES_KEY set: {bool(os.environ.get("GOOGLE_PLACES_KEY"))}')

---
## 1. Venue Wisata — OpenStreetMap (Overpass API)

Harvest venue wisata dalam batas administratif DKI Jakarta (admin_level=4).  
Filter: `tourism`, `leisure`, `historic`, `amenity:place_of_worship`.

Legal & gratis — OSM berlisensi ODbL, boleh dipakai dengan atribusi.

In [ ]:
print('Filter kategori wisata OSM:')
for k, v in config.TOURISM_FILTERS.items():
    print(f'  {k}: {v}')
print(f'\nBoundary: {config.JAKARTA_AREA_NAME} (admin_level={config.JAKARTA_AREA_ADMIN_LEVEL})')

In [ ]:
# [RUN] Harvest venue OSM (~2-3 menit, butuh internet)
import csv, time, requests

HEADERS = {'User-Agent': 'JakartaWisataRec/1.0'}

def build_query(bbox, key, values, area_name=None, area_admin_level=None):
    if values is None:
        sel = f'["{key}"]["name"]'
    else:
        regex = '|'.join(values)
        sel = f'["{key}"~"^({regex})$"]["name"]'
    if area_name:
        area_def = f'area["name"="{area_name}"]["admin_level"="{area_admin_level}"]->.a;'
        body = f'node{sel}(area.a);way{sel}(area.a);'
        return f'[out:json][timeout:120];{area_def}({body});out center tags;'
    s, w, n, e = bbox
    box = f'({s},{w},{n},{e})'
    return f'[out:json][timeout:120];(node{sel}{box};way{sel}{box};);out center tags;'

def fetch_osm(query, retries=2):
    for url in config.OVERPASS_URLS:
        for attempt in range(retries):
            try:
                resp = requests.post(url, data={'data': query}, headers=HEADERS, timeout=180)
            except requests.RequestException as exc:
                print(f'  {url} error: {exc}'); break
            if resp.status_code == 200:
                return resp.json()
            print(f'  {url} status {resp.status_code}, retry {attempt+1}/{retries}...')
            time.sleep(5 * (attempt + 1))
    raise RuntimeError('Semua endpoint Overpass gagal.')

def derive_category(tags):
    for key in ('tourism', 'leisure', 'historic', 'amenity'):
        if key in tags:
            return f'{key}:{tags[key]}'
    return 'unknown'

def parse_osm(elements):
    rows = []
    for el in elements:
        tags = el.get('tags', {})
        name = tags.get('name')
        if not name: continue
        if el['type'] == 'node':
            lat, lon = el.get('lat'), el.get('lon')
        else:
            center = el.get('center', {})
            lat, lon = center.get('lat'), center.get('lon')
        if lat is None or lon is None: continue
        rows.append({
            'venue_id': f'{el["type"]}/{el["id"]}',
            'name': name,
            'venue_category': derive_category(tags),
            'latitude': lat, 'longitude': lon,
            'opening_hours': tags.get('opening_hours', ''),
            'website': tags.get('website') or tags.get('contact:website', ''),
            'wikipedia': tags.get('wikipedia', ''),
            'wikidata': tags.get('wikidata', ''),
            'osm_url': f'https://www.openstreetmap.org/{el["type"]}/{el["id"]}',
            'maps_url': f'https://www.google.com/maps/search/?api=1&query={lat},{lon}',
        })
    return rows

if not os.path.exists(config.RAW_CSV):
    print('Mulai harvest OSM...')
    os.makedirs('data/raw', exist_ok=True)
    all_elements = []
    for key, values in config.TOURISM_FILTERS.items():
        print(f'Query: {key} ...')
        try:
            q = build_query(config.JAKARTA_BBOX, key, values,
                            config.JAKARTA_AREA_NAME, config.JAKARTA_AREA_ADMIN_LEVEL)
            data = fetch_osm(q)
        except Exception as exc:
            print(f'  area query gagal ({exc}), fallback bbox...')
            q = build_query(config.JAKARTA_BBOX, key, values)
            data = fetch_osm(q)
        els = data.get('elements', [])
        print(f'  {len(els)} elemen')
        all_elements.extend(els)
        time.sleep(8)
    rows = parse_osm(all_elements)
    with open(config.RAW_CSV, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        writer.writeheader(); writer.writerows(rows)
    print(f'Tersimpan -> {config.RAW_CSV} ({len(rows)} venue)')
else:
    print(f'File sudah ada: {config.RAW_CSV} — skip harvest')

In [ ]:
# [STATS] OSM
if os.path.exists(config.RAW_CSV):
    osm_raw = pd.read_csv(config.RAW_CSV)
    print(f'Total venue OSM: {len(osm_raw):,}')
    print()
    print('Top 10 kategori:')
    print(osm_raw['venue_category'].value_counts().head(10).to_string())

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].scatter(osm_raw['longitude'], osm_raw['latitude'], s=2, alpha=0.3, color='steelblue')
    axes[0].set_title(f'Sebaran {len(osm_raw):,} venue OSM — DKI Jakarta')
    axes[0].set_xlabel('Longitude'); axes[0].set_ylabel('Latitude')

    osm_raw['venue_category'].value_counts().head(8).plot(
        kind='barh', ax=axes[1], color='steelblue',
        title='Top 8 kategori venue OSM')
    axes[1].invert_yaxis()
    axes[1].set_xlabel('Jumlah venue')
    plt.tight_layout()
    plt.show()

---
## 2. Check-in Nyata — Massive-STEPS-Jakarta (HuggingFace)

Dataset penelitian publik: 412.100 check-in nyata pengguna Foursquare-style, 2012–2018.  
`checkin_count` per venue = proxy popularitas nyata (bukan simulasi).

Repo: `CRUISEResearchGroup/Massive-STEPS-Jakarta`

In [ ]:
# [RUN] Download dari HuggingFace (~beberapa menit tergantung koneksi)
from huggingface_hub import hf_hub_download

checkins_path = 'data/raw/jakarta_checkins_raw.csv'

if not os.path.exists(checkins_path):
    print(f'Download {config.STEPS_REPO_ID} / {config.STEPS_CHECKINS_FILENAME} ...')
    os.makedirs('data/raw', exist_ok=True)
    path = hf_hub_download(
        repo_id=config.STEPS_REPO_ID,
        filename=config.STEPS_CHECKINS_FILENAME,
        repo_type='dataset'
    )
    df_steps = pd.read_csv(path)
    df_steps.to_csv(checkins_path, index=False)
    print(f'Total check-in mentah: {len(df_steps):,}')
    print(f'Tersimpan -> {checkins_path}')
else:
    print(f'File sudah ada: {checkins_path} — skip download')

In [ ]:
# [STATS] STEPS
if os.path.exists(checkins_path):
    checkins = pd.read_csv(checkins_path)
    print(f'Total check-in (mentah)        : 412.100')
    print(f'Setelah drop null lat/lon/name : {len(checkins):,}')
    print(f'Kolom: {list(checkins.columns)}')
    print()
    print('Null per kolom:')
    null_cols = checkins.isnull().sum()
    print(null_cols[null_cols > 0].to_string())

    # Venue unik
    venues_steps = checkins.groupby('venue_id').agg(
        name=('name','first'),
        venue_category=('venue_category','first'),
        latitude=('latitude','first'),
        longitude=('longitude','first'),
        checkin_count=('venue_id','count')
    ).reset_index()
    print(f'\nVenue unik: {len(venues_steps):,}')

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    venues_steps['venue_category'].value_counts().head(12).plot(
        kind='barh', ax=axes[0], color='darkorange',
        title=f'Top 12 kategori STEPS (dari {len(venues_steps):,} venue unik)')
    axes[0].invert_yaxis()
    axes[0].set_xlabel('Jumlah venue')

    venues_steps['checkin_count'].clip(upper=500).hist(
        bins=30, ax=axes[1], color='darkorange', edgecolor='white')
    axes[1].set_title('Distribusi checkin_count per venue (cap 500)')
    axes[1].set_xlabel('Check-in count')
    axes[1].set_ylabel('Jumlah venue')
    plt.tight_layout()
    plt.show()

---
## 3. Venue Wisata Tambahan — Google Places Nearby Search

Massive-STEPS (2012–2018) tidak merata — venue terkenal yang tidak populer di Foursquare mungkin tidak tercatat.  
Solusi: Google Places Nearby Search per kategori × 21 anchor point cover seluruh DKI Jakarta.

Hasil disimpan ke `data/raw/venues_google_raw.csv` (belum difilter noise — itu di preprocessing).

In [ ]:
# [RUN] Google Places Nearby Search venue wisata (butuh GOOGLE_PLACES_KEY)
import json
from datetime import date

GVENUE_CACHE_DIR = 'data/raw/google_venues_cache'
GVENUE_OUT = 'data/raw/venues_google_raw.csv'
BASE_URL = 'https://places.googleapis.com/v1'
DAYS_ID = ['Senin','Selasa','Rabu','Kamis','Jumat','Sabtu','Minggu']
GDAY_TO_ID = {0:'Minggu',1:'Senin',2:'Selasa',3:'Rabu',4:'Kamis',5:'Jumat',6:'Sabtu'}

CATEGORY_SEARCHES = [
    ('Zoo',                        ['zoo'],                                         'Kebun Binatang'),
    ('Aquarium',                   ['aquarium'],                                    'Akuarium'),
    ('Theme Park',                 ['amusement_park','water_park'],                 'Taman Hiburan'),
    ('Museum',                     ['museum'],                                      'Museum'),
    ('Art Museum',                 ['art_gallery'],                                 'Galeri Seni'),
    ('Historic Site',              ['tourist_attraction','cultural_landmark'],      'Situs Bersejarah'),
    ('History Museum',             ['history_museum'],                              'Museum Sejarah'),
    ('Science Museum',             ['museum'],                                      'Museum Sains'),
    ('Temple',                     ['hindu_temple','buddhist_temple'],              'Vihara/Pura'),
    ('Monument / Landmark',        ['monument','tourist_attraction'],               'Monumen'),
    ('Beach',                      ['beach'],                                       'Pantai'),
    ('Theme Park Ride / Attraction',['tourist_attraction','amusement_park'],        'Wahana Wisata'),
]

SEARCH_ANCHORS = [
    (-6.1754,106.8272,'Menteng'),(-6.1600,106.8450,'Kemayoran'),(-6.1950,106.8200,'Tanah Abang'),
    (-6.2400,106.7900,'Kebayoran Baru'),(-6.2600,106.8100,'Mampang'),(-6.2900,106.8300,'Pasar Minggu'),
    (-6.2200,106.8700,'Pancoran'),(-6.3000,106.7700,'Cilandak'),
    (-6.1200,106.8800,'Pademangan'),(-6.1000,106.8500,'Penjaringan'),(-6.0900,106.9200,'Cilincing'),
    (-6.1400,106.7800,'Grogol Utara'),(-6.1700,106.7600,'Grogol'),(-6.2000,106.7400,'Kebon Jeruk'),
    (-6.1500,106.7200,'Cengkareng'),(-6.1900,106.7000,'Kembangan'),
    (-6.2100,106.8900,'Jatinegara'),(-6.2500,106.9200,'Kramat Jati'),
    (-6.1800,106.9400,'Pulo Gadung'),(-6.2800,106.8900,'Pasar Rebo'),
    (-5.7200,106.5700,'Kepulauan Seribu'),
]

def search_nearby_google(lat, lon, google_types, api_key, radius=5000):
    headers = {
        'Content-Type': 'application/json', 'X-Goog-Api-Key': api_key,
        'X-Goog-FieldMask': ','.join([
            'places.id','places.displayName','places.formattedAddress',
            'places.location','places.rating','places.userRatingCount',
            'places.priceLevel','places.regularOpeningHours','places.types',
            'places.editorialSummary','places.nationalPhoneNumber',
        ]),
    }
    body = {
        'includedTypes': google_types,
        'locationRestriction': {'circle': {'center': {'latitude': lat, 'longitude': lon}, 'radius': float(radius)}},
        'maxResultCount': 20,
    }
    try:
        r = requests.post(f'{BASE_URL}/places:searchNearby', headers=headers, json=body, timeout=20)
        return r.json().get('places', []) if r.status_code == 200 else []
    except: return []

def parse_hours_google(periods):
    result = {d: None for d in DAYS_ID}
    for p in periods:
        day_id = GDAY_TO_ID.get(p.get('open',{}).get('day'))
        if not day_id: continue
        o, c = p.get('open',{}), p.get('close',{})
        result[day_id] = (f'{o.get("hour",0):02d}:{o.get("minute",0):02d}',
                          f'{c.get("hour",0):02d}:{c.get("minute",0):02d}')
    return result

if not os.path.exists(GVENUE_OUT):
    api_key = os.environ.get('GOOGLE_PLACES_KEY', '')
    if not api_key:
        print('Set GOOGLE_PLACES_KEY di environment terlebih dahulu.')
    else:
        os.makedirs(GVENUE_CACHE_DIR, exist_ok=True)
        all_places = {}
        for fsq_cat, gtypes, label in CATEGORY_SEARCHES:
            print(f'=== {fsq_cat} ({label}) ===')
            cat_count = 0
            for lat, lon, area in SEARCH_ANCHORS:
                cache_key = f"{fsq_cat.replace('/',' ').replace(' ','_')}_{area.replace(' ','_')}"
                cache_path = os.path.join(GVENUE_CACHE_DIR, f'{cache_key}.json')
                if os.path.exists(cache_path):
                    with open(cache_path, encoding='utf-8') as f: places = json.load(f)
                else:
                    places = search_nearby_google(lat, lon, gtypes, api_key)
                    with open(cache_path, 'w', encoding='utf-8') as f: json.dump(places, f, ensure_ascii=False)
                    time.sleep(0.3)
                for p in places:
                    pid = p.get('id')
                    if pid and pid not in all_places:
                        all_places[pid] = (p, fsq_cat); cat_count += 1
            print(f'  Venue baru: {cat_count}')

        rows = []
        for pid, (p, fsq_cat) in all_places.items():
            loc = p.get('location', {})
            hours = parse_hours_google(p.get('regularOpeningHours',{}).get('periods',[]))
            row = {
                'place_id': pid, 'name': p.get('displayName',{}).get('text',''),
                'venue_category': fsq_cat, 'address': p.get('formattedAddress',''),
                'latitude': loc.get('latitude'), 'longitude': loc.get('longitude'),
                'google_rating': p.get('rating'), 'google_rating_count': p.get('userRatingCount'),
                'description': p.get('editorialSummary',{}).get('text',''),
                'google_types': ','.join(p.get('types',[])),
                'hours_source': 'google_places', 'data_collected': date.today().strftime('%Y-%m-%d'),
            }
            for day in DAYS_ID:
                val = hours.get(day)
                row[f'{day}_buka'] = val[0] if val else 'Tutup'
                row[f'{day}_tutup'] = val[1] if val else 'Tutup'
            rows.append(row)

        df_gv = pd.DataFrame(rows)
        df_gv = df_gv.dropna(subset=['latitude','longitude'])
        df_gv = df_gv[(df_gv['latitude'].between(-6.40,-5.10)) & (df_gv['longitude'].between(106.50,107.10))]
        df_gv.to_csv(GVENUE_OUT, index=False)
        print(f'\nTotal venue unik: {len(df_gv)}')
        print(f'Tersimpan -> {GVENUE_OUT}')
else:
    print(f'File sudah ada: {GVENUE_OUT} — skip harvest')

In [ ]:
# [STATS] Google Places venue
if os.path.exists(GVENUE_OUT):
    gv = pd.read_csv(GVENUE_OUT)
    print(f'Total venue Google raw: {len(gv)} (sebelum filter noise di preprocessing)')
    print(f'Rating rata-rata: {gv["google_rating"].mean():.2f}')
    print()
    print('Distribusi kategori:')
    print(gv['venue_category'].value_counts().to_string())

    gv['venue_category'].value_counts().plot(
        kind='barh', figsize=(8,5), color='teal',
        title=f'Distribusi kategori Google Places raw ({len(gv)} venue)')
    plt.gca().invert_yaxis()
    plt.xlabel('Jumlah venue')
    plt.tight_layout()
    plt.show()

---
## 4. Hotel — Google Places Nearby Search

Dipisah dari venue wisata — peran hotel = titik start/end itinerary, bukan venue yang diskor/direkomendasikan.  
21 anchor point cover seluruh DKI + Kepulauan Seribu, radius 5km, kategori `hotel/lodging/motel/guest_house/hostel`.

In [ ]:
# [RUN] Google Places Nearby Search hotel (butuh GOOGLE_PLACES_KEY)
HOTEL_CACHE_DIR = 'data/processed/google_hotel_cache'
HOTEL_OUT = 'data/processed/hotels_google.csv'

HOTEL_ANCHORS = [
    (-6.1754,106.8272,'Menteng'),(-6.1600,106.8450,'Kemayoran'),(-6.1950,106.8200,'Tanah Abang'),
    (-6.2400,106.7900,'Kebayoran Baru'),(-6.2600,106.8100,'Mampang'),(-6.2900,106.8300,'Pasar Minggu'),
    (-6.2200,106.8700,'Pancoran'),(-6.3000,106.7700,'Cilandak'),
    (-6.1200,106.8800,'Pademangan'),(-6.1000,106.8500,'Penjaringan'),(-6.0900,106.9200,'Cilincing'),
    (-6.1400,106.7800,'Grogol Utara'),(-6.1700,106.7600,'Grogol'),(-6.2000,106.7400,'Kebon Jeruk'),
    (-6.1500,106.7200,'Cengkareng'),(-6.1900,106.7000,'Kembangan'),
    (-6.2100,106.8900,'Jatinegara'),(-6.2500,106.9200,'Kramat Jati'),
    (-6.1800,106.9400,'Pulo Gadung'),(-6.2800,106.8900,'Pasar Rebo'),
    (-5.7200,106.5700,'Kepulauan Seribu'),
]

def search_hotels_nearby(lat, lon, api_key, radius=5000):
    headers = {
        'Content-Type': 'application/json', 'X-Goog-Api-Key': api_key,
        'X-Goog-FieldMask': ','.join([
            'places.id','places.displayName','places.formattedAddress',
            'places.location','places.rating','places.userRatingCount',
            'places.priceLevel','places.regularOpeningHours',
            'places.websiteUri','places.nationalPhoneNumber',
            'places.types','places.editorialSummary',
        ]),
    }
    body = {
        'includedTypes': ['hotel','lodging','motel','guest_house','hostel'],
        'locationRestriction': {'circle': {'center': {'latitude': lat, 'longitude': lon}, 'radius': float(radius)}},
        'maxResultCount': 20,
    }
    try:
        r = requests.post(f'{BASE_URL}/places:searchNearby', headers=headers, json=body, timeout=20)
        return r.json().get('places', []) if r.status_code == 200 else []
    except: return []

if not os.path.exists(HOTEL_OUT):
    api_key = os.environ.get('GOOGLE_PLACES_KEY', '')
    if not api_key:
        print('Set GOOGLE_PLACES_KEY di environment terlebih dahulu.')
    else:
        os.makedirs(HOTEL_CACHE_DIR, exist_ok=True)
        os.makedirs('data/processed', exist_ok=True)
        all_hotels = {}
        for lat, lon, area in HOTEL_ANCHORS:
            cache_path = os.path.join(HOTEL_CACHE_DIR, f"{area.replace(' ','_')}.json")
            if os.path.exists(cache_path):
                with open(cache_path, encoding='utf-8') as f: places = json.load(f)
                print(f'{area}: {len(places)} hotel (cache)')
            else:
                print(f'Search hotel: {area} ...')
                places = search_hotels_nearby(lat, lon, api_key)
                with open(cache_path, 'w', encoding='utf-8') as f: json.dump(places, f, ensure_ascii=False)
                print(f'  {len(places)} hotel')
                time.sleep(0.5)
            for p in places:
                pid = p.get('id')
                if pid and pid not in all_hotels: all_hotels[pid] = p

        rows = []
        for pid, p in all_hotels.items():
            loc = p.get('location', {})
            hours = parse_hours_google(p.get('regularOpeningHours',{}).get('periods',[]))
            row = {
                'place_id': pid, 'name': p.get('displayName',{}).get('text',''),
                'address': p.get('formattedAddress',''),
                'latitude': loc.get('latitude'), 'longitude': loc.get('longitude'),
                'google_rating': p.get('rating'), 'google_rating_count': p.get('userRatingCount'),
                'price_level': p.get('priceLevel'), 'website': p.get('websiteUri',''),
                'phone': p.get('nationalPhoneNumber',''),
                'description': p.get('editorialSummary',{}).get('text',''),
                'google_types': ','.join(p.get('types',[])),
            }
            for day in DAYS_ID:
                val = hours.get(day)
                row[f'{day}_buka'] = val[0] if val else 'Tutup'
                row[f'{day}_tutup'] = val[1] if val else 'Tutup'
            rows.append(row)

        df_h = pd.DataFrame(rows).dropna(subset=['latitude','longitude'])
        df_h = df_h[(df_h['latitude'].between(-6.40,-5.10)) & (df_h['longitude'].between(106.50,107.10))]
        df_h.to_csv(HOTEL_OUT, index=False)
        print(f'\nTotal hotel unik: {len(df_h)}')
        print(f'Tersimpan -> {HOTEL_OUT}')
else:
    print(f'File sudah ada: {HOTEL_OUT} — skip harvest')

In [ ]:
# [STATS] Hotel
if os.path.exists(HOTEL_OUT):
    hotels = pd.read_csv(HOTEL_OUT)
    print(f'Total hotel Google raw: {len(hotels)}')
    print(f'Rating rata-rata      : {hotels["google_rating"].mean():.2f}')
    print(f'Hotel dengan rating   : {hotels["google_rating"].notna().sum()}/{len(hotels)}')

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].scatter(hotels['longitude'], hotels['latitude'], s=4, alpha=0.5, color='crimson')
    axes[0].set_title(f'Sebaran {len(hotels)} hotel Google Places — DKI Jakarta')
    axes[0].set_xlabel('Longitude'); axes[0].set_ylabel('Latitude')

    hotels['google_rating'].dropna().hist(bins=20, ax=axes[1], color='coral', edgecolor='white')
    axes[1].set_title('Distribusi rating hotel (Google)')
    axes[1].set_xlabel('Rating'); axes[1].set_ylabel('Jumlah hotel')
    plt.tight_layout()
    plt.show()

    print('\nTop 10 hotel (rating tertinggi):')
    print(hotels.nlargest(10, 'google_rating')[['name','google_rating','google_rating_count','address']].to_string(index=False))

---
## Ringkasan Semua Sumber Data

In [ ]:
# [STATS] Summary bar chart
summary = {}
if os.path.exists(config.RAW_CSV):
    summary['OSM\n(venues_osm_raw)'] = len(pd.read_csv(config.RAW_CSV))
if os.path.exists('data/raw/jakarta_checkins_raw.csv'):
    summary['STEPS\ncheck-in'] = len(pd.read_csv('data/raw/jakarta_checkins_raw.csv'))
if os.path.exists('data/raw/venues_google_raw.csv'):
    summary['Google Places\nvenue raw'] = len(pd.read_csv('data/raw/venues_google_raw.csv'))
if os.path.exists(HOTEL_OUT):
    summary['Google Places\nhotel raw'] = len(pd.read_csv(HOTEL_OUT))

if summary:
    fig, ax = plt.subplots(figsize=(12, 5))
    colors = ['steelblue','darkorange','teal','crimson']
    bars = ax.bar(summary.keys(), summary.values(), color=colors[:len(summary)])
    ax.set_title('Jumlah data per sumber (sebelum preprocessing)')
    ax.set_ylabel('Jumlah row')
    for bar, val in zip(bars, summary.values()):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+2000,
                f'{val:,}', ha='center', va='bottom', fontsize=9)
    plt.tight_layout()
    plt.show()

print('\n=== Ringkasan ===')
for k, v in summary.items():
    print(f'  {k.replace(chr(10)," "):30s}: {v:,}')